#### (1) 회귀 모델 성능 비교

In [ ]:
import pandas as pd 
import matplotlib.pyplot as plt 
from sklearn.model_selection import train_test_split 
from sklearn.metrics import mean_squared_error, mean_absolute_error, r2_score 
from sklearn.linear_model import Ridge 
from sklearn.ensemble import RandomForestRegressor, GradientBoostingRegressor

# 데이터 로드 및 전처리 
df = pd.read_csv('../datasets/Clean_Dataset.csv') # 데이터 파일 로드 
df = df.drop(['flight', 'departure_time', 'stops', 'arrival_time'], axis=1) # 학습에 필요 없는 문자열 열제거 
df = pd.get_dummies(df, columns=['airline', 'source_city', 'destination_city', 'class'], drop_first=True)

# 범주형 데이터 원-핫 인코딩 
X = df.drop('price', axis=1) # 종속 변수인 'price'를 제외한 독립 변수 
y = df['price'] # 종속 변수 설정

# 데이터 분리 
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42) # 학습용(80%) 과 테스트용(20%) 데이터 분리

# 모델 정의 
models = { 
    'Ridge Regression': Ridge(alpha=1.0), # Ridge 회귀 모델 (L2 정규화) 
    'Random Forest': RandomForestRegressor(n_estimators=100, max_depth=10, random_state=42), # 랜덤 포레스트 모델 
    'Gradient Boosting': GradientBoostingRegressor(n_estimators=100, learning_rate=0.1, max_depth=5, random_state=42) # 그래디언트 부스트 모델 
}

# 성능 평가 
results = [] 
for name, model in models.items(): # 모델별 성능 비교 루프 
    model.fit(X_train, y_train) # 모델 학습 
    predictions = model.predict(X_test) # 테스트 데이터 예측 
    mse = mean_squared_error(y_test, predictions) # 평균 제곱 오차 계산 
    mae = mean_absolute_error(y_test, predictions) # 평균 절대 오차 계산 
    r2 = r2_score(y_test, predictions) # 결정 계수 계산 
    results.append({'Model': name, 'MSE': mse, 'MAE': mae, 'R²': r2}) # 결과 저장

# 결과를 데이터프레임으로 변환 
results_df = pd.DataFrame(results) # 성능 결과를 데이터프레임으로 변환

# 시각화 
fig, ax = plt.subplots(1, 2, figsize=(14, 6)) # 두 개의 그래프를 위한 figure 생성 
results_df.sort_values(by='MSE', ascending=True).plot.bar(x='Model', y='MSE', ax=ax[0], color='skyblue', legend=False) # MSE 막대 그래프 생성 
ax[0].set_title('Mean Squared Error (MSE)') # MSE 그래프 제목 설정 
ax[0].set_ylabel('MSE') # MSE y축 레이블 설정

results_df.sort_values(by='R²', ascending=False).plot.bar(x='Model', y='R²', ax=ax[1], color='orange', legend=False) # R² 막대 그래프 생성 
ax[1].set_title('R² Score') # R² 그래프 제목 설정 
ax[1].set_ylabel('R²') # R² y축 레이블 설정

plt.tight_layout() # 그래프 간격 정리 
plt.show() # 그래프 출력

# 결과 출력 
print("Regression Model Results:\n", results_df) # 성능 결과 출력

#### (2) 분류 모델 성능 비교

In [2]:
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score, roc_auc_score 
from sklearn.linear_model import LogisticRegression 
from sklearn.ensemble import RandomForestClassifier, GradientBoostingClassifier 
from sklearn.svm import SVC

# 데이터 로드 및 전처리 
df = pd.read_csv('../datasets/Clean_Dataset.csv') # 데이터 파일 로드 
df = df.drop(['flight', 'departure_time', 'stops', 'arrival_time'], axis=1) # 학습에 필요 없는 문자열 제거

df = pd.get_dummies(df, columns=['airline', 'source_city', 'destination_city', 'class'], drop_first=True) # 범주형 데이터 원-핫 인코딩 
X = df.drop('price', axis=1) # 종속 변수 'price'를 제외한 독립 변수 
y = (df['price'] > df['price'].median()).astype(int) # 중간값 기준 이진 분류

# 데이터 분리 
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42) # 학습용(80%)과 테스트용(20%) 데이터 분리

# 데이터 스케일링 
from sklearn.preprocessing import StandardScaler 
scaler = StandardScaler() 
X_train_scaled = scaler.fit_transform(X_train) # 학습 데이터를 스케일링 
X_test_scaled = scaler.transform(X_test) # 테스트 데이터를 스케일링

# 모델 정의 
models = { 
    'Logistic Regression': LogisticRegression(max_iter=10, solver='saga'), 
    'Random Forest': RandomForestClassifier(n_estimators=10, max_depth=10, random_state= 42), 
    'Gradient Boosting': GradientBoostingClassifier(n_estimators=10, learning_rate=0.1, max_depth=5, random_state=42), 
    'SVM': SVC(kernel='linear', probability=False) # 확률 계산 비활성화 
}

results = [] 
for name, model in models.items():
    model.fit(X_train_scaled, y_train) # 스케일링된 데이터로 학습 
    predictions = model.predict(X_test_scaled) # 스케일링된 데이터로 예측 
    probabilities = model.predict_proba(X_test_scaled)[:, 1] if hasattr(model, "predict_proba") else None 
    accuracy = accuracy_score(y_test, predictions) 
    precision = precision_score(y_test, predictions) 
    recall = recall_score(y_test, predictions) 
    f1 = f1_score(y_test, predictions) 
    auc = roc_auc_score(y_test, probabilities) if probabilities is not None else "N/A" 
    results.append({'Model': name, 'Accuracy': accuracy, 'Precision': precision, 'Recall': recall, 'F1 Score': f1, 'ROC AUC': auc})

# 결과를 데이터프레임으로 변환 
results_df = pd.DataFrame(results) # 성능 결과를 데이터프레임으로 변환 
results_df

# 성능 평가 결과를 시각화 
fig, ax = plt.subplots(2, 2, figsize=(18, 12)) # 2행 3열 서브 플롯 생성

# Accuracy 그래프 
results_df.set_index('Model')['Accuracy'].plot.bar(ax=ax[0, 0], color='blue', legend=False) 
ax[0, 0].set_title("Accuracy") 
ax[0, 0].set_ylabel("Score")

# Precision 그래프 
results_df.set_index('Model')['Precision'].plot.bar(ax=ax[0, 1], color='green', legend=False) 
ax[0, 1].set_title("Precision") 
ax[0, 1].set_ylabel("Score")

# Recall 그래프 
results_df.set_index('Model')['Recall'].plot.bar(ax=ax[1, 0], color='orange', legend=False) 
ax[1, 0].set_title("Recall") 
ax[1, 0].set_ylabel("Score")

# F1 Score 그래프 
results_df.set_index('Model')['F1 Score'].plot.bar(ax=ax[1, 1], color='purple', legend=False) 
ax[1, 1].set_title("F1 Score") 
ax[1, 1].set_ylabel("Score")

# 빈 서브 플롯 처리 
ax[1, 2].axis('off') # 남는 한 칸을 비활성화 
plt.tight_layout() # 그래프 간격 조정 
plt.show()

NameError: name 'pd' is not defined